In [29]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Folder name created in Drive
MASTER_PATH = '/content/drive/MyDrive/Canteen_AI_Project'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### **Filter** **Data**

In [30]:
import os
import shutil


INPUT_DIR = os.path.join(MASTER_PATH, 'Raw_Data', 'Images')
FILTERED_DIR = os.path.join(MASTER_PATH, 'Filtered_Data')

def run_forced_filter():
    if not os.path.exists(FILTERED_DIR):
        os.makedirs(FILTERED_DIR)

    for food in SELECTED_CLASSES:
        source = os.path.join(INPUT_DIR, food)
        dest = os.path.join(FILTERED_DIR, food)

        if os.path.exists(source):
            os.makedirs(dest, exist_ok=True)
            # Copy all files from Raw to Filtered
            files = os.listdir(source)
            for f in files[:20]: # Grab 20 to be safe
                shutil.copy(os.path.join(source, f), os.path.join(dest, f))
            print(f"Filtered & Copied: {food}")
        else:
            print(f"Warning: Could not find {food}")

run_forced_filter()

Filtered & Copied: Chicken Bun
Filtered & Copied: Chicken Cutlet
Filtered & Copied: Chicken Rolls
Filtered & Copied: Chicken Pastry
Filtered & Copied: Chicken Rotti
Filtered & Copied: Pizza - Cheese Lovers
Filtered & Copied: Milk Rice
Filtered & Copied: Iced Coffee


### **Clean Images**

In [31]:
import cv2
import os
import shutil

CLEANED_DIR = os.path.join(MASTER_PATH, 'Cleaned_Data')

def advanced_quality_filter():
    if not os.path.exists(CLEANED_DIR):
        os.makedirs(CLEANED_DIR)

    for food in SELECTED_CLASSES:
        source_folder = os.path.join(FILTERED_DIR, food)
        target_folder = os.path.join(CLEANED_DIR, food)
        os.makedirs(target_folder, exist_ok=True)

        all_images = [f for f in os.listdir(source_folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        scored_images = []

        print(f" Advanced Cleaning: {food}...")

        for img_name in all_images:
            path = os.path.join(source_folder, img_name)
            img = cv2.imread(path)
            if img is None: continue

            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

            # Laplacian (Edges)
            lap_score = cv2.Laplacian(gray, cv2.CV_64F).var()

            # Sobel (Texture/Detail)
            sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
            sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
            sobel_score = (sobelx**2 + sobely**2).mean()

            # Combine scores (Final Quality Score)
            final_score = lap_score + (sobel_score / 10)

            scored_images.append((final_score, img_name))

        # Sort by best score
        scored_images.sort(key=lambda x: x[0], reverse=True)

        # Take the top 10
        top_10 = scored_images[:10]

        for score, img_name in top_10:
            shutil.copy(os.path.join(source_folder, img_name), os.path.join(target_folder, img_name))

        print(f"{food}: Cleaned and saved the 10 highest-detail images.")

advanced_quality_filter()

 Advanced Cleaning: Chicken Bun...
Chicken Bun: Cleaned and saved the 10 highest-detail images.
 Advanced Cleaning: Chicken Cutlet...
Chicken Cutlet: Cleaned and saved the 10 highest-detail images.
 Advanced Cleaning: Chicken Rolls...
Chicken Rolls: Cleaned and saved the 10 highest-detail images.
 Advanced Cleaning: Chicken Pastry...
Chicken Pastry: Cleaned and saved the 10 highest-detail images.
 Advanced Cleaning: Chicken Rotti...
Chicken Rotti: Cleaned and saved the 10 highest-detail images.
 Advanced Cleaning: Pizza - Cheese Lovers...
Pizza - Cheese Lovers: Cleaned and saved the 10 highest-detail images.
 Advanced Cleaning: Milk Rice...
Milk Rice: Cleaned and saved the 10 highest-detail images.
 Advanced Cleaning: Iced Coffee...
Iced Coffee: Cleaned and saved the 10 highest-detail images.


### **Resize** **Images**

In [32]:
from PIL import Image
import os


FINAL_DIR = os.path.join(MASTER_PATH, 'Final_Canteen_Dataset')

def run_resize_final():
    if not os.path.exists(FINAL_DIR):
        os.makedirs(FINAL_DIR)

    for food in SELECTED_CLASSES:
        source_folder = os.path.join(CLEANED_DIR, food)
        target_folder = os.path.join(FINAL_DIR, food)
        os.makedirs(target_folder, exist_ok=True)

        images = os.listdir(source_folder)

        for img_name in images:
            path = os.path.join(source_folder, img_name)
            try:
                with Image.open(path) as img:
                    # Convert to RGB (removes transparency issues)
                    img = img.convert("RGB")

                    # Resize to 640x640 using high-quality downsampling
                    img = img.resize((640, 640), Image.Resampling.LANCZOS)

                    # Save to the Final folder
                    img.save(os.path.join(target_folder, img_name))
            except Exception as e:
                print(f"Skipping {img_name}: {e}")

        print(f"Resized: {food} (10 images) are now exactly 640x640.")

run_resize_final()

Resized: Chicken Bun (10 images) are now exactly 640x640.
Resized: Chicken Cutlet (10 images) are now exactly 640x640.
Resized: Chicken Rolls (10 images) are now exactly 640x640.
Resized: Chicken Pastry (10 images) are now exactly 640x640.
Resized: Chicken Rotti (10 images) are now exactly 640x640.
Resized: Pizza - Cheese Lovers (10 images) are now exactly 640x640.
Resized: Milk Rice (10 images) are now exactly 640x640.
Resized: Iced Coffee (10 images) are now exactly 640x640.
